# Earnings Quality — Reported vs Underlying Profitability
## Greek Systemic Banks (2022–2024)

**What equity analysts actually care about**: recurring, sustainable earnings — not one-offs.
This notebook strips identified non-recurring items and restates ROE and C/I on an underlying basis.

## Identified One-Off Items

| Bank | Year | Item | Pre-tax Impact (€m) | Source |
|------|------|------|--------------------:|--------|
| Eurobank | 2022 | Excess trading / fair value gains on bond portfolio (incl. Eurolife insurance income). 'Other operating income' of €1,136m vs €283m normalised avg (2023–2024 avg). Strip €853m excess. | +853 (one-off gain) | Annual Report 2022, Note 8 (Other Operating Income); comparison to 2023/2024 normalised levels |
| Eurobank | 2024 | Hellenic Bank consolidation effect: total assets +€21.3bn on first consolidation (Q1 2024). Balance sheet denominator inflation overstates leverage-driven ROE in transition year. **NIM adjustment only**: denominator inflated mid-year; no adjustable P&L one-off visible in summary IS. | Noted, no direct P&L adjustment | Annual Report 2024, Note 4 (Business Combinations) |

> **Methodology**: One-off amounts are expressed pre-tax. After-tax impact uses each bank-year's effective tax rate derived from `net_profit / profit_before_tax`. Adjusted operating income = reported − one-off (pre-tax). Adjusted net profit = reported − (one-off × (1 − effective tax rate)).
>
> **Limitation**: without access to the granular note disclosures in the PDFs (pages 180–220 of Eurobank 2022 Annual Report), the exact €853m figure is an approximation based on IS normalisation. Treat as directional, not precise.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

COLORS = {
    'Eurobank':    '#0067B1',
    'Alpha Bank':  '#E2231A',
    'Piraeus Bank':'#F7A600',
    'NBG':         '#003087',
}
BANKS = list(COLORS.keys())
DATA_DIR = Path('../02_Banking_Sector_Dashboard/data/processed')

kpis = pd.read_csv(DATA_DIR / 'kpis_final.csv')
is_long = pd.read_csv(DATA_DIR / 'income_statement_final.csv')

# Pivot IS to get Profit Before Tax per bank-year
is_piv = is_long.pivot_table(index=['bank','year'], columns='metric', values='value', aggfunc='first').reset_index()
pbt_col = 'Profit before tax'
df = kpis.merge(is_piv[['bank','year',pbt_col]].rename(columns={pbt_col:'profit_before_tax'}), on=['bank','year'])

print(f'Loaded {len(df)} bank-year observations.')

In [ ]:
# ── Define one-off items table ────────────────────────────────────────────────
# Format: (bank, year, pretax_oneoff_gain)
# Positive = one-off gain that inflated reported earnings (strip it out)
# Negative = one-off charge that depressed reported earnings (add it back)

ONE_OFFS = [
    # Eurobank 2022: excess other operating income vs 2023-2024 normalised average
    # Other income = OpIncome - NII - Fees = 3135 - 1551 - 448 = 1136m
    # Normalised avg (2023: 293m, 2024: 274m) = 283m
    # Excess = 1136 - 283 = 853m pre-tax
    ('Eurobank', 2022, 853.0, 'Excess trading/FV gains vs normalised level'),
]

one_off_df = pd.DataFrame(ONE_OFFS, columns=['bank','year','pretax_oneoff','description'])
print('One-off table:')
print(one_off_df.to_string(index=False))

In [ ]:
# ── Compute underlying metrics ────────────────────────────────────────────────
df = df.merge(one_off_df[['bank','year','pretax_oneoff']], on=['bank','year'], how='left')
df['pretax_oneoff'] = df['pretax_oneoff'].fillna(0.0)

# Effective tax rate from reported figures
df['eff_tax_rate'] = 1 - (df['net_profit'] / df['profit_before_tax'])

# Adjusted operating income and net profit
df['adj_operating_income'] = df['operating_income'] - df['pretax_oneoff']
df['adj_net_profit']       = df['net_profit'] - df['pretax_oneoff'] * (1 - df['eff_tax_rate'])

# Adjusted KPIs
df['adj_roe']            = df['adj_net_profit'] / df['equity'] * 100
df['adj_cost_to_income'] = df['operating_expenses'].abs() / df['adj_operating_income'] * 100

print('\n── Reported vs Underlying (adjusted) ──')
cols = ['bank','year','roe','adj_roe','cost_to_income','adj_cost_to_income','pretax_oneoff']
out = df[cols].copy()
for c in ['roe','adj_roe','cost_to_income','adj_cost_to_income']:
    out[c] = out[c].round(1)
print(out.to_string(index=False))

In [ ]:
# ── Chart 1: Reported vs Underlying ROE ───────────────────────────────────────
fig1 = make_subplots(
    rows=1, cols=2,
    subplot_titles=['<b>ROE: Reported vs Underlying (%)</b>', '<b>Cost/Income: Reported vs Underlying (%)</b>'],
    horizontal_spacing=0.12,
)

YEARS = [2022, 2023, 2024]

for bank in BANKS:
    bdf = df[df['bank'] == bank].sort_values('year')

    # Reported ROE
    fig1.add_trace(go.Bar(
        x=[f'{bank}<br>{y}' for y in YEARS],
        y=bdf['roe'].values,
        name=f'{bank} Reported',
        marker_color=COLORS[bank],
        opacity=0.9,
        showlegend=True,
        legendgroup=bank,
    ), row=1, col=1)

    # Underlying ROE (dot overlay)
    fig1.add_trace(go.Scatter(
        x=[f'{bank}<br>{y}' for y in YEARS],
        y=bdf['adj_roe'].values,
        mode='markers',
        marker=dict(color='white', size=10, symbol='diamond', line=dict(color=COLORS[bank], width=2)),
        name=f'{bank} Underlying',
        showlegend=True,
        legendgroup=f'{bank}_adj',
    ), row=1, col=1)

    # Reported C/I
    fig1.add_trace(go.Bar(
        x=[f'{bank}<br>{y}' for y in YEARS],
        y=bdf['cost_to_income'].values,
        name=f'{bank} Reported',
        marker_color=COLORS[bank],
        opacity=0.9,
        showlegend=False,
        legendgroup=bank,
    ), row=1, col=2)

    # Underlying C/I
    fig1.add_trace(go.Scatter(
        x=[f'{bank}<br>{y}' for y in YEARS],
        y=bdf['adj_cost_to_income'].values,
        mode='markers',
        marker=dict(color='white', size=10, symbol='diamond', line=dict(color=COLORS[bank], width=2)),
        name=f'{bank} Underlying',
        showlegend=False,
        legendgroup=f'{bank}_adj',
    ), row=1, col=2)

fig1.add_annotation(
    text='◆ = Underlying (one-offs stripped)',
    xref='paper', yref='paper', x=0.5, y=-0.18,
    showarrow=False, font=dict(size=12, color='white'),
)

fig1.update_layout(
    title_text='<b>Earnings Quality: Reported vs Underlying</b> | Bars = Reported, Diamonds = Underlying',
    title_font_size=16,
    template='plotly_dark',
    height=460,
    barmode='group',
    paper_bgcolor='#0f1117',
    plot_bgcolor='#0f1117',
    legend=dict(orientation='h', y=-0.22, x=0.5, xanchor='center', font_size=10),
)
fig1.show()

In [ ]:
# ── Chart 2: Impact Bridge — one-off effect on ROE ────────────────────────────
adj_rows = df[df['pretax_oneoff'] != 0][['bank','year','roe','adj_roe','pretax_oneoff','description']].copy()

if len(adj_rows) == 0:
    print('No one-offs to visualise.')
else:
    for _, r in adj_rows.iterrows():
        gap = r['roe'] - r['adj_roe']
        print(f"  {r['bank']} {int(r['year'])}: Reported ROE {r['roe']:.1f}% → Underlying {r['adj_roe']:.1f}% "
              f"(gap {gap:+.1f}pp | one-off {r['pretax_oneoff']:+.0f}m pre-tax)")
        print(f"  Item: {r['description']}")

print('\n── Key Insight ──')
print('Eurobank 2022 reported ROE of 20.0% includes unusually high non-recurring income.')
eurobank_22 = df[(df['bank']=='Eurobank') & (df['year']==2022)].iloc[0]
print(f'Underlying ROE (ex one-offs): ~{eurobank_22["adj_roe"]:.1f}%')
print(f'This is still sector-leading but significantly below the headline 20.0%.')
print(f'Peer comparison on underlying ROE in 2022 is more representative of earnings power.')

In [ ]:
# ── Final validation ──────────────────────────────────────────────────────────
assert len(df) == 12, 'Expected 12 bank-year rows.'
assert df['eff_tax_rate'].between(0, 1).all(), 'Effective tax rate outside [0,1].'

# For rows with no one-off, reported ≈ underlying (within rounding of stored ROE)
no_oneoff = df[df['pretax_oneoff'] == 0]
roe_diff  = (no_oneoff['roe'] - no_oneoff['adj_roe']).abs()
assert roe_diff.max() < 0.1, f'ROE adjustment on zero one-off rows exceeds 0.1pp: {roe_diff.max():.4f}'

# Eurobank 2022 underlying ROE must be lower than reported
euro22 = df[(df['bank']=='Eurobank') & (df['year']==2022)].iloc[0]
assert euro22['adj_roe'] < euro22['roe'], 'Expected underlying ROE < reported for Eurobank 2022'

# Underlying C/I must be higher (denominator reduced, costs unchanged)
assert euro22['adj_cost_to_income'] > euro22['cost_to_income'], \
    'Expected underlying C/I > reported when stripping income one-off'

print('✅ All checks passed.')
print(f"   Eurobank 2022: Reported ROE {euro22['roe']:.1f}% → Underlying {euro22['adj_roe']:.1f}% "
      f"(Δ {euro22['roe']-euro22['adj_roe']:+.1f}pp)")
print(f"   Eurobank 2022: Reported C/I {euro22['cost_to_income']:.1f}% → Underlying {euro22['adj_cost_to_income']:.1f}%")

In [ ]:
# ── Final validation ──────────────────────────────────────────────────────────
assert len(df) == 12, 'Expected 12 bank-year rows.'
assert df['eff_tax_rate'].between(0, 1).all(), 'Effective tax rate outside [0,1].'

# For rows with no one-off, reported = underlying
no_oneoff = df[df['pretax_oneoff'] == 0]
roe_diff  = (no_oneoff['roe'] - no_oneoff['adj_roe']).abs()
assert roe_diff.max() < 0.01, f'Non-zero ROE adjustment on zero one-off rows: {roe_diff.max():.4f}'

# Eurobank 2022 underlying ROE must be lower than reported
euro22 = df[(df['bank']=='Eurobank') & (df['year']==2022)].iloc[0]
assert euro22['adj_roe'] < euro22['roe'], 'Expected underlying ROE < reported for Eurobank 2022'

# Underlying C/I must be higher (denominator reduced, costs unchanged)
assert euro22['adj_cost_to_income'] > euro22['cost_to_income'], \
    'Expected underlying C/I > reported when stripping income one-off'

print('✅ All checks passed.')
print(f"   Eurobank 2022: Reported ROE {euro22['roe']:.1f}% → Underlying {euro22['adj_roe']:.1f}% "
      f"(Δ {euro22['roe']-euro22['adj_roe']:+.1f}pp)")
print(f"   Eurobank 2022: Reported C/I {euro22['cost_to_income']:.1f}% → Underlying {euro22['adj_cost_to_income']:.1f}%")